<a href="https://colab.research.google.com/github/jcmachicao/knowledge_engineering/blob/main/U3__swarm_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import math

In [4]:
class Path:
    """Representa un camino posible explorado por una hormiga."""
    def __init__(self, nodes):
        self.nodes = nodes
        self.length = len(nodes)

    def is_solution(self):
        # Aquí podrías definir un criterio real, por ahora un ejemplo:
        return len(self.nodes) > 3 and self.nodes[0] == 0 and self.nodes[-1] == 9

In [9]:
class SharedEnvironment:
    """Entorno compartido donde se almacenan feromonas."""
    def __init__(self):
        self.pheromones = {}  # {(nodo_a, nodo_b): cantidad}

    def deposit_pheromone(self, edge, strength):
        self.pheromones[edge] = self.pheromones.get(edge, 0) + strength

    def evaporate_pheromones(self, rate=0.1):
        for edge in list(self.pheromones.keys()):
            self.pheromones[edge] *= (1 - rate)
            if self.pheromones[edge] < 1e-5:
                del self.pheromones[edge]

    def get_pheromone(self, edge):
        return self.pheromones.get(edge, 0)

    def get_strongest_pheromone_trail(self):
        if not self.pheromones:
            return None
        edge = max(self.pheromones, key=self.pheromones.get)
        return edge, self.pheromones[edge]

In [10]:
class SimpleAgent:
    """Hormiga individual con decisiones locales."""
    def __init__(self, id):
        self.id = id
        self.current_node = random.randint(0, 9)

    def explore_locally(self, environment):
        # Explora un camino aleatorio guiado por feromonas
        path = [self.current_node]
        for _ in range(random.randint(3, 7)):
            next_node = random.randint(0, 9)
            edge = (path[-1], next_node)
            pheromone = environment.get_pheromone(edge)
            # Probabilidad influida por feromonas
            if random.random() < (0.7 + 0.3 * min(1, pheromone)):
                path.append(next_node)
            else:
                # Exploración aleatoria
                path.append(random.randint(0, 9))
        return Path(path)

    def deposit_pheromone(self, path, strength):
        # Deposita feromonas a lo largo del camino recorrido
        for i in range(len(path.nodes) - 1):
            edge = (path.nodes[i], path.nodes[i + 1])
            self.environment.deposit_pheromone(edge, strength)

In [11]:
class SwarmCognitiveSystem:
    """
    Sistema cognitivo sin controlador central
    Inspirado en: hormigas, abejas, peces, aves
    """
    def __init__(self, n_agents=100, max_iterations=50):
        self.agents = [SimpleAgent(id=i) for i in range(n_agents)]
        self.environment = SharedEnvironment()
        self.max_iterations = max_iterations
        for a in self.agents:
            a.environment = self.environment  # Conexión compartida

    def simulate_ant_colony_optimization(self, problem=None):
        """
        Hormigas resolviendo problemas complejos
        SIN LÍDER, SIN PLAN CENTRAL
        """
        for iteration in range(self.max_iterations):
            for ant in self.agents:
                path = ant.explore_locally(self.environment)
                if path.is_solution():
                    ant.deposit_pheromone(path, strength=1 / path.length)
            self.environment.evaporate_pheromones(rate=0.1)

        best_path = self.environment.get_strongest_pheromone_trail()
        return best_path

In [14]:
for i in range(20):
  swarm = SwarmCognitiveSystem(n_agents=i, max_iterations=100)
  solution = swarm.simulate_ant_colony_optimization()
  print("Posible solución emergente:", solution)

Posible solución emergente: None
Posible solución emergente: None
Posible solución emergente: ((4, 0), 0.03654232816007434)
Posible solución emergente: None
Posible solución emergente: ((8, 9), 0.0712532964852856)
Posible solución emergente: None
Posible solución emergente: None
Posible solución emergente: None
Posible solución emergente: ((3, 9), 0.2522838593117946)
Posible solución emergente: None
Posible solución emergente: None
Posible solución emergente: ((0, 4), 0.18240846762358875)
Posible solución emergente: ((5, 9), 0.16211152318449765)
Posible solución emergente: ((7, 9), 0.04305850691486571)
Posible solución emergente: ((0, 2), 0.00791377226848119)
Posible solución emergente: ((0, 3), 0.18246037874468557)
Posible solución emergente: ((8, 1), 0.15118860335205178)
Posible solución emergente: ((0, 8), 0.07845264902250004)
Posible solución emergente: ((0, 9), 0.026789702351555773)
Posible solución emergente: ((0, 9), 0.1484974184043021)


In [15]:
import random
from collections import Counter
random.seed(42)  # reproducibilidad

def run_experiments(n_runs=20, n_agents=30, max_iterations=100):
    success_count = 0
    top_edges = []
    strengths = []

    for i in range(n_runs):
        # opcional: reseed por corrida con i si quieres variación controlada
        random.seed(42 + i)

        swarm = SwarmCognitiveSystem(n_agents=n_agents, max_iterations=max_iterations)
        result = swarm.simulate_ant_colony_optimization()
        print(f"Run {i}: {result}")

        if result is None:
            continue
        edge, strength = result
        top_edges.append(edge)
        strengths.append(strength)
        success_count += 1

    edge_counts = Counter(top_edges)
    print("\nResumen:")
    print(f"Corridas totales: {n_runs}, corridas con solución: {success_count}")
    print("Top edges (frecuencia):", edge_counts.most_common(10))
    if strengths:
        print("Fuerza media del mejor edge:", sum(strengths)/len(strengths))
        print("Máximo:", max(strengths), "Mínimo:", min(strengths))

# ejemplo
run_experiments(n_runs=20, n_agents=30, max_iterations=100)

Run 0: ((0, 4), 0.2512032622553881)
Run 1: ((7, 9), 0.2887007948937246)
Run 2: ((5, 9), 0.08751114771934099)
Run 3: ((6, 9), 0.31195332245654717)
Run 4: ((0, 6), 0.3713496971259541)
Run 5: ((0, 2), 0.38764069562187303)
Run 6: ((0, 7), 0.19558209780000008)
Run 7: ((9, 9), 0.3780423813893077)
Run 8: ((0, 6), 0.05294343396719546)
Run 9: ((0, 9), 0.3352968458984173)
Run 10: ((0, 3), 0.3664941364913141)
Run 11: ((7, 9), 0.18885603124717962)
Run 12: ((2, 6), 0.13334647712458322)
Run 13: ((0, 4), 0.20723265864297502)
Run 14: ((0, 2), 0.17260858700398105)
Run 15: ((0, 8), 0.2952481010203479)
Run 16: ((0, 9), 0.09393041846936057)
Run 17: ((0, 9), 0.3859893966914057)
Run 18: ((2, 9), 0.32926274261382343)
Run 19: ((2, 5), 0.2025)

Resumen:
Corridas totales: 20, corridas con solución: 20
Top edges (frecuencia): [((0, 9), 3), ((0, 4), 2), ((7, 9), 2), ((0, 6), 2), ((0, 2), 2), ((5, 9), 1), ((6, 9), 1), ((0, 7), 1), ((9, 9), 1), ((0, 3), 1)]
Fuerza media del mejor edge: 0.25178461142163594
Máximo: 0